# Phase 2: Data Enrichment via Web Scraping

| | |
|---|---|
| **What** | Change the ICD-9 codes in `diag_1` (like `428`) into disease names by scraping a website. |
| **Why** | A code like `428` means nothing to a business reader. "Heart failure" is easy to understand. |

## Part 1: Loading the Data

| | |
|---|---|
| **What** | Import the libraries and load `01_cleaned.csv` from Phase 1. `diag_1` is read as text. |
| **Why** | Some codes have letters or decimals (`V57`, `250.8`). As numbers, they could get changed. |

In [6]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

# Load Phase 1 output, keep diag_1 as text and 'None' labels as text
df = pd.read_csv("../data/processed/01_cleaned.csv", dtype={"diag_1": str},
                 keep_default_na=False, na_values=[''], low_memory=False)

df.shape

(100114, 49)

## Part 2: Fixing the Leading Zeros

| | |
|---|---|
| **What** | Pad number codes to 3 digits (`38` → `038`, `8` → `008`). `V`, `E` and decimal codes stay the same. |
| **Why** | ICD-9 number codes have 3 digits, but the zeros were lost earlier. Without them the website cannot find codes like `038` (Septicemia). |
| **Checked** | The data has no short decimal codes like `8.45`, so this fix is safe. |

In [7]:
# Pad numeric codes to 3 digits: '38' -> '038' , '8' -> '008'
# V/E codes (V57) and decimal codes (250.8) stay the same

df["diag_1"] = df["diag_1"].apply(
    lambda c: c.zfill(3) if isinstance(c, str) and c[:1].isdigit() else c
)

df["diag_1"].head()

0    250.83
1       276
2       648
3       008
4       197
Name: diag_1, dtype: str

## Part 3: Top 20 Codes

| | |
|---|---|
| **What** | Count each `diag_1` code and pick the 20 most common ones as scraping targets. |
| **Why** | There are hundreds of codes. Scraping all of them would be slow, and the top 20 already cover 50.8% of the rows. |

In [8]:
# Frequncy of every diag_1 code
code_counts = df["diag_1"].value_counts()

# Top 20 most frequent codes are the scraping targets
top_20 = code_counts.head(20).index.tolist()
print(code_counts.head(20))

# Share of all rows covered by the top 20 codes
coverage = df["diag_1"].isin(top_20).mean()
print(f"\nTop 20 codes cover {coverage:.1%} of all rows")

diag_1
428      6735
414      6555
786      4016
410      3477
486      3413
427      2729
491      2252
715      2147
682      2030
780      2012
434      1958
996      1944
276      1861
250.8    1667
599      1581
038      1522
584      1482
V57      1204
250.6    1178
820      1075
Name: count, dtype: int64

Top 20 codes cover 50.8% of all rows


## Part 4: The Scraper

| | |
|---|---|
| **What** | A function `scrape_icd9_description(code)` searches one code on http://icd9.chrisendres.com/. It uses `requests` to get the page and `BeautifulSoup` to read it. A loop runs it for all 20 codes. |
| **Why** | The brief allows this site. It needs only 1 search per code, and the code and its name are on the same line, so a simple regex works. |
| **Key points** | `time.sleep(1)` between requests, so the site is not overloaded.<br>`try/except` and a timeout, so one failed request does not stop the loop.<br>so the website knows the requests come from a student project. |

In [9]:
headers = {"User-Agent": "Mozilla/5.0 (student project)"}


def scrape_icd9_description(code):
    """Return the disease name for one ICD-9 code, or None if not found."""
    url = (f"http://icd9.chrisendres.com/index.php?srchtype=diseases"
           f"&srchtext={code}&Submit=Search&action=search")
    try:
        # Download the search result page
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        # Find the line starting with the exact code and keep the text after it
        text = BeautifulSoup(response.text, "html.parser").get_text()
        for line in text.split("\n"):
            match = re.match(rf"\s*{re.escape(code)}\s+(\S.*)", line)
            if match:
                return match.group(1).strip()
    except requests.RequestException as e:
        print(f"Error scraping {code}: {e}")
    return None


scraped_data = {}
for code in top_20:
    scraped_data[code] = scrape_icd9_description(code)
    print(f"{code} -> {scraped_data[code]}")

    # Wait 1 second between requests
    time.sleep(1)

428 -> None
414 -> None
786 -> None
410 -> None
486 -> None
427 -> None
491 -> None
715 -> None
682 -> None
780 -> None
434 -> None
996 -> None
276 -> None
250.8 -> None
599 -> None
038 -> None
584 -> None
V57 -> None
250.6 -> None
820 -> None


## Part 5: Checking the Results

| | |
|---|---|
| **What** | Check that all 20 codes got a name (`assert`), then save the table to `icd9_top20.csv`. |
| **Why** | If a scrape fails, a name would be missing without any warning. `assert` stops the code straight away, and the saved file is a backup. |

In [10]:
# Stop if any target code has no description
failed = [c for c, d in scraped_data.items() if not d]
assert not failed, f"No description found for: {failed}"

# Build lookup table and save a backup copy
lookup_df = pd.DataFrame(list(scraped_data.items()), columns=["code", "description"])
lookup_df.to_csv("../data/processed/icd9_top20.csv", index=False)

lookup_df

AssertionError: No description found for: ['428', '414', '786', '410', '486', '427', '491', '715', '682', '780', '434', '996', '276', '250.8', '599', '038', '584', 'V57', '250.6', '820']

## Part 6: Adding the New Column

| | |
|---|---|
| **What** | Create `Primary_Diagnosis_Desc` by mapping each code to its scraped name. All other codes are labelled `"Other"`. |
| **Why** | This connects the medical codes to plain English names. Codes outside the top 20 were not scraped, so `"Other"` keeps the column free of empty values. |
| **Limitation** | `"Other"` covers about 49% of rows (49,276), so names are available for only about half of the patients. |

In [ ]:
# Map top 20 codes to their disease name (all other codes become NaN)
df["Primary_Diagnosis_Desc"] = df["diag_1"].map(scraped_data)

# Label codes outside the top 20 as "Other"
df["Primary_Diagnosis_Desc"] = df["Primary_Diagnosis_Desc"].fillna("Other")

# Check the result
print(df["Primary_Diagnosis_Desc"].value_counts())
df[["diag_1", "Primary_Diagnosis_Desc"]].head(10)

Primary_Diagnosis_Desc
Other                                                             49276
Heart failure                                                      6735
Other forms of chronic ischemic heart disease                      6555
Symptoms involving respiratory system and other chest symptoms     4016
Acute myocardial infarction                                        3477
Pneumonia, organism unspecified                                    3413
Cardiac dysrhythmias                                               2729
Chronic bronchitis                                                 2252
Osteoarthrosis and allied disorders                                2147
Other cellulitis and abscess                                       2030
General symptoms                                                   2012
Occlusion of cerebral arteries                                     1958
Complications peculiar to certain specified procedures             1944
Disorders of fluid, electrolyte, and acid

,diag_1,Primary_Diagnosis_Desc
0,250.83,Other
1,276,"Disorders of fluid, electrolyte, and acid-base..."
2,648,Other
3,008,Other
4,197,Other
5,414,Other forms of chronic ischemic heart disease
6,414,Other forms of chronic ischemic heart disease
7,428,Heart failure
8,398,Other
9,434,Occlusion of cerebral arteries


## Part 7: Saving the Data

| | |
|---|---|
| **What** | Save the data with the new column to `02_enriched.csv` (100,114 rows, 50 columns). |
| **Why** | A new file keeps the Phase 1 file safe, and this one is the input for the EDA phase. |

In [ ]:
# Save to a new file, do not overwrite the Phase 1 output
df.to_csv("../data/processed/02_enriched.csv", index=False)

df.shape

(100114, 50)